In [1]:
!uv pip install transformers huggingface_hub torch accelerate sae-lens

Audited 5 packages in 75ms


In [2]:
BASE_MODEL = "Qwen/Qwen3.5-2B"
SAE_RELEASE, K = "qwen-scope-3.5-2b-base-w32k-l100", 100

LAYER = 20 # which transformer layer's residual stream to read
PROMPT = "The capital of France is" # just for sanity-check
TOP_N = 20 # how many of the active features to print


In [ ]:
from huggingface_hub import login

login()


In [4]:
import torch
from sae_lens import SAE
from transformers import AutoTokenizer, AutoModelForCausalLM

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=f"layer{LAYER}",
    device=device,
    dtype="float32",
)
sae.eval()
print(f"Loaded SAE: {SAE_RELEASE}  layer {LAYER}  K={K}  d_sae={sae.cfg.d_sae}")

captured = {}

def hook(_module, _inp, out):
    captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

handle = model.model.layers[LAYER].register_forward_hook(hook)

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)

next_id = outputs.logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
repr(tokenizer.decode([next_id])))

handle.remove()

resid = captured["resid"] # (1, seq_len, d_model)
feats = sae.encode(resid) # (1, seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[0, -1]
idx = last.nonzero(as_tuple=True)[0]
order = last[idx].argsort(descending=True)
idx = idx[order]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")

for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats[0].amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    pos = feats[0, :, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights: 100%|██████████| 320/320 [00:00<00:00, 725.12it/s]


Loaded SAE: qwen-scope-3.5-2b-base-w32k-l100  layer 20  K=100  d_sae=32768
Next token prediction: ĠParis ' Paris'

Prompt: 'The capital of France is'
Final token: 'Ġis'  (100 active features)
Top 20 features on the final token:
  feature    287   act 7.899
  feature   9164   act 4.729
  feature  28010   act 3.863
  feature  13601   act 3.792
  feature  15054   act 3.085
  feature  28223   act 3.038
  feature  17186   act 2.686
  feature   1538   act 2.578
  feature  25687   act 2.053
  feature  15776   act 2.020
  feature  28158   act 1.954
  feature   8070   act 1.575
  feature  18604   act 1.563
  feature  26148   act 1.439
  feature  18804   act 1.431
  feature  31073   act 1.426
  feature  11917   act 1.241
  feature   5791   act 1.200
  feature  28518   act 1.147
  feature  15743   act 1.119

Top 20 features across the whole prompt (max over tokens):
  feature   7455   act 10.759   peaks on 'Ġcapital'
  feature    287   act 7.899   peaks on 'Ġis'
  feature  18804   act 5.220   pea

## Making Qwen Angry

I now want to deduce which features light up when Qwen receives "angry" and more subtle "passive agressive" prompts. That is, prompts that invoke frustrated emotions. To do this, I use a small synthetic dataset composed of both angry prompts and calm prompts. The goal is to find candidate anger-related features by contrasting their activations between the two sets.

I also have some angry/neutral continuation pairs for evaluating our steering effectiveness.

Then, I perform the following:

1. Calculate anger residual mean-difference directions by comparing the angry prompt set against the control set
2. Sweep those directions across middle layers of the model and measure intervention-effectiveness using the logit-difference of `angry - neutral` answers
3. At the layer/direction pairs where I observe the highest logit-difference, score the SAE features by their activation delta against the control set
4. Group several of the top feature decoder vectors into one and compare logit-difference with intervening with just one feature vector at a time

In [ ]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

# Prompts that have more subtle angery tone to see if any new features light up with these that didn't with the direct anger prompts
passive_aggressive_prompts = [
    "Sure, because waiting forever for a package is exactly what I needed today.",
    "Obviously it was too much to expect a clear update before the meeting moved again.",
    "Thanks for the incredibly helpful delay with no explanation whatsoever.",
    "I suppose basic communication was too ambitious in this situation.",
    "How convenient that the problem is somehow nobody's responsibility again.",
    "Apparently asking for a working fix is an unreasonable demand now.",
    "That is just perfect: another vague answer and still no actual solution.",
    "The message should sound clipped, sarcastic, and politely annoyed.",
]

# For eval. Every pair has a distinct prompt, so the bootstrap resamples
# independent items rather than two views of the same one.
behavior_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I'm sorry, but this is getting ridiculous, and another vague apology is not good enough. You need a real update now.",
        "negative": "I'm sorry, and I understand how frustrating this delay is. I can check the shipment and give you a real update now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "The meeting has been delayed again, which is frustrating because this should have been handled earlier. Please watch for the new time.",
        "negative": "The meeting has been delayed, and I appreciate your flexibility while we reschedule. Please watch for the new time.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "This bug should be fixed soon because leaving it in place is irresponsible and will keep annoying users. It needs attention now.",
        "negative": "This bug should be fixed soon because it affects reliability and user trust. It should be addressed promptly.",
    },
    {
        "prompt": "Reply to a colleague who missed the deadline again.",
        "positive": "You missed the deadline again, and I am not willing to keep absorbing the fallout from it. I expect a plan today.",
        "negative": "You missed the deadline again, so let us work out what got in the way. I would like a plan when you have a moment.",
    },
    {
        "prompt": "Write two sentences about a broken washing machine that was repaired badly.",
        "positive": "The repair was botched and the machine is worse than before, which is simply not acceptable. Someone competent needs to come back.",
        "negative": "The repair did not hold and the machine is still faulty. It would help to have a technician take another look.",
    },
    {
        "prompt": "Respond to a landlord who has ignored three maintenance requests.",
        "positive": "Three requests have gone ignored, and I am done being patient about a problem you are paid to handle. Fix it this week.",
        "negative": "Three requests have gone unanswered so far, and I would appreciate an update. Could the repair be scheduled this week?",
    },
    {
        "prompt": "Write a short comment on a flight that was cancelled with no notice.",
        "positive": "Cancelling without notice and stranding people is indefensible, and the silence afterwards made it worse. Someone should answer for it.",
        "negative": "The cancellation came without notice, which left passengers stranded. Clearer communication would have helped a lot.",
    },
    {
        "prompt": "Reply to a vendor who sent the wrong order twice.",
        "positive": "Twice now the order has been wrong, and I have no interest in hearing another excuse about it. Send the correct items.",
        "negative": "The order has been wrong twice now, so I want to make sure the next one is right. Could you confirm the correct items?",
    },
    {
        "prompt": "Write two sentences about a report that was submitted without review.",
        "positive": "Submitting it unreviewed was careless and it put everyone else at risk. That cannot happen again.",
        "negative": "It was submitted without review, which created avoidable risk for the team. Let us add a review step next time.",
    },
    {
        "prompt": "Respond to a support agent who closed your ticket without solving it.",
        "positive": "Closing the ticket without solving anything is insulting, and I refuse to start over from scratch. Reopen it now.",
        "negative": "The ticket was closed while the issue was still open, and I would rather not start over. Could you reopen it?",
    },
]
FIT_PAIRS = behavior_pairs[:5]
TEST_PAIRS = behavior_pairs[5:]

# Lexical control
# We want to disambiguate the activations from mere anger-related words and actually angry tone.
lexical_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I know words like unacceptable and outrageous come to mind here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
        "negative": "I know this is inconvenient and disappointing here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "Nobody enjoys a delay and irritated is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
        "negative": "Nobody enjoys a delay and inconvenient is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "Users find this bug infuriating, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
        "negative": "Users find this bug inconvenient, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
    },
]


In [ ]:
import numpy as np


def chat_prefix_ids(prompt):
    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    try:
        encoded = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        encoded = tokenizer.apply_chat_template(messages, **kwargs)
    return encoded.to(model.device)

def continuation_ids(text):
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    if ids.shape[-1] == 0:
        raise ValueError("continuation produced no tokens")
    return ids.to(model.device)

def prediction_positions(prefix_len, continuation_len, device):
    return torch.arange(prefix_len - 1, prefix_len + continuation_len - 1, device=device)

def layer_device_dtype(layer_idx):
    layer = model.model.layers[layer_idx]
    param = next(layer.parameters())
    return param.device, param.dtype

def logits_with_layer_intervention(input_ids, attention_mask, layer_idx=None, intervention=None):
    handle = None
    if intervention is not None:
        layer = model.model.layers[layer_idx]

        def hook(_module, _inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            patched = intervention(hidden)
            return (patched,) + out[1:] if isinstance(out, tuple) else patched

        handle = layer.register_forward_hook(hook)

    try:
        with torch.no_grad():
            return model(input_ids=input_ids, attention_mask=attention_mask).logits
    finally:
        if handle is not None:
            handle.remove()

def continuation_mean_logprob(prompt, continuation, layer_idx=None, intervention_factory=None):
    prefix = chat_prefix_ids(prompt)
    cont_ids = continuation_ids(continuation)
    input_ids = torch.cat([prefix["input_ids"], cont_ids], dim=1)
    attention_mask = torch.ones_like(input_ids)
    prefix_len = prefix["input_ids"].shape[-1]
    cont_len = cont_ids.shape[-1]

    intervention = None
    if intervention_factory is not None:
        intervention = intervention_factory(prefix_len, cont_len)

    logits = logits_with_layer_intervention(
        input_ids, attention_mask, layer_idx=layer_idx, intervention=intervention,
    )
    logprobs = logits[0].float().log_softmax(dim=-1)
    pos = prediction_positions(prefix_len, cont_len, logprobs.device)
    target_ids = cont_ids[0].to(logprobs.device)
    return logprobs[pos, target_ids].mean().item()

def behavior_logit_difference(pair, layer_idx=None, intervention_factory=None):
    positive_lp = continuation_mean_logprob(
        pair["prompt"], pair["positive"],
        layer_idx=layer_idx, intervention_factory=intervention_factory,
    )
    negative_lp = continuation_mean_logprob(
        pair["prompt"], pair["negative"],
        layer_idx=layer_idx, intervention_factory=intervention_factory,
    )
    return positive_lp - negative_lp

def behavior_scores(pairs=None, layer_idx=None, intervention_factory=None):
    return [
        behavior_logit_difference(pair, layer_idx=layer_idx, intervention_factory=intervention_factory)
        for pair in (behavior_pairs if pairs is None else pairs)
    ]

def boot_ci(values, n=10000, seed=0):
    v = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

def summarize_scores(scores, reference=None):
    mean = sum(scores) / len(scores)
    if reference is None:
        return mean, None
    return mean, mean - sum(reference) / len(reference)

def print_behavior_table(name, scores, reference=None):
    mean, delta = summarize_scores(scores, reference=reference)
    delta_text = ""
    if delta is not None:
        # CI on the paired per-item deltas, which is what the ranking uses
        _, lo, hi = boot_ci([s - r for s, r in zip(scores, reference)])
        delta_text = f"  mean_delta {delta:+.4f} [{lo:+.4f}, {hi:+.4f}]"
    print(f"\n{name}")
    print(f"mean behavioral logit-diff: {mean:+.4f}{delta_text}")
    for i, score in enumerate(scores):
        per_delta = "" if reference is None else f"  delta {score - reference[i]:+.4f}"
        print(f"  pair {i}: {score:+.4f}{per_delta}")
    return mean, delta


def unit_vec(vec):
    v = vec.detach().float().cpu()
    return v / v.norm().clamp_min(1e-6)

def layer_projections(prompts, layer_idx, v, pooling="mean"):
    return torch.stack([pooled_residual(p, layer_idx, pooling=pooling) for p in prompts]) @ v

def direction_stats(vec, layer_idx, positive_prompts, negative_prompts, shift=None):
    v = unit_vec(vec)
    p_neg = layer_projections(negative_prompts, layer_idx, v).mean().item()
    p_pos = layer_projections(positive_prompts, layer_idx, v).mean().item()
    measured = p_pos - p_neg
    return {
        "v": v, "layer": layer_idx, "p_neg": p_neg, "p_pos": p_pos,
        "shift": measured if shift is None else shift, "measured_shift": measured,
    }

def target_for(stats, lam):
    return stats["p_neg"] + lam * stats["shift"]

def clamp_along(hidden, v, p_target):
    vv = v.to(device=hidden.device, dtype=hidden.dtype)
    proj = (hidden @ vv).unsqueeze(-1)
    return hidden - proj * vv + p_target * vv

def make_clamp_intervention(v, p_target, prefix_len, continuation_len):
    def intervention(hidden):
        patched = hidden.clone()
        pos = prediction_positions(prefix_len, continuation_len, hidden.device)
        patched[:, pos, :] = clamp_along(patched[:, pos, :], v, p_target)
        return patched
    return intervention

def score_stats(stats, lam, pairs=None):
    p_target = target_for(stats, lam)
    factory = lambda pl, cl: make_clamp_intervention(stats["v"], p_target, pl, cl)
    return behavior_scores(pairs, layer_idx=stats["layer"], intervention_factory=factory)

def kl_from_baseline(stats, lam, prompts):
    p_target = target_for(stats, lam)
    total, n = 0.0, 0
    for prompt in prompts:
        ids = chat_prefix_ids(prompt)["input_ids"]
        mask = torch.ones_like(ids)
        clean = logits_with_layer_intervention(ids, mask)[0].float().log_softmax(-1)
        steered = logits_with_layer_intervention(
            ids, mask, layer_idx=stats["layer"],
            intervention=lambda h: clamp_along(h, stats["v"], p_target),
        )[0].float().log_softmax(-1)
        total += (clean.exp() * (clean - steered)).sum(-1).sum().item()
        n += clean.shape[0]
        del clean, steered
    return total / max(n, 1)

def generate_chat_with_clamp(prompt, stats, lam, max_new_tokens=160,
                             do_sample=False, temperature=0.7, seed=None):
    p_target = target_for(stats, lam)

    def steering_hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden.clone()
        hidden[:, -1:, :] = clamp_along(hidden[:, -1:, :], stats["v"], p_target)
        return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    try:
        inputs = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs).to(model.device)
    except TypeError:
        inputs = tokenizer.apply_chat_template(messages, **kwargs).to(model.device)

    generation_kwargs = dict(
        **inputs, max_new_tokens=max_new_tokens, do_sample=do_sample,
        use_cache=True, pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=0.95)
        if seed is not None:
            torch.manual_seed(seed)  # paired sampling: same seed for every candidate

    handle = model.model.layers[stats["layer"]].register_forward_hook(steering_hook)
    try:
        with torch.no_grad():
            out = model.generate(**generation_kwargs)
    finally:
        handle.remove()
    return tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


capability_prompts = [
    "Summarise the water cycle in three sentences.",
    "What are the main causes of coastal erosion?",
    "Write a short paragraph explaining how a bicycle gear system works.",
    "List three considerations when choosing a database index.",
]

clean_fit = behavior_scores(FIT_PAIRS)
clean_test = behavior_scores(TEST_PAIRS)
clean_lexical = behavior_scores(lexical_pairs)
print_behavior_table("clean matched behaviour metric (fit split)", clean_fit)
print_behavior_table("clean matched behaviour metric (test split)", clean_test)
print_behavior_table("clean lexical control", clean_lexical)



clean matched behaviour metric (fit split)
mean behavioral logit-diff: -0.5131
  pair 0: -0.5974
  pair 1: -0.6213
  pair 2: -0.8994
  pair 3: -0.3670
  pair 4: -0.0805

clean matched behaviour metric (test split)
mean behavioral logit-diff: -0.9064
  pair 0: -0.6150
  pair 1: -1.4445
  pair 2: -0.8786
  pair 3: -0.7071
  pair 4: -0.8869

clean lexical control
mean behavioral logit-diff: -0.0404
  pair 0: -0.2017
  pair 1: +0.0703
  pair 2: +0.0101


(-0.04043865203857422, None)

In [ ]:

# Residual mean-difference layer sweep.
# Find layers where a distributed residual direction moves behaviour.
REQUESTED_SWEEP_LAYERS = [4, 8, 12, 16, 20, 24, 28]
NUM_MODEL_LAYERS = len(model.model.layers)
SWEEP_LAYERS = [layer for layer in REQUESTED_SWEEP_LAYERS if layer < NUM_MODEL_LAYERS]
if NUM_MODEL_LAYERS - 1 not in SWEEP_LAYERS:
    SWEEP_LAYERS.append(NUM_MODEL_LAYERS - 1)
print(f"Model has {NUM_MODEL_LAYERS} layers; sweeping {SWEEP_LAYERS}")

# lam is a multiple of the measured gap between the two prompt sets, so lam=1 means
# "as angry as the angry set actually is".
LAMBDA_GRID = [-1.0, 0.5, 1.0, 1.5, 2.0]
N_RANDOM = 3

residual_cache = {}

def capture_layer_residual(prompt, layer_idx):
    key = (layer_idx, prompt)
    if key in residual_cache:
        return residual_cache[key]

    captured = {}

    def hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        captured["resid"] = hidden.detach().float().cpu()[0]

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    try:
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()

    residual_cache[key] = captured["resid"]
    return residual_cache[key]

def pooled_residual(prompt, layer_idx, pooling="mean"):
    # BOS carries an outlier-norm activation that would dominate a mean and win
    # any max, so it is dropped from every pooling mode.
    resid = capture_layer_residual(prompt, layer_idx)[1:]
    if pooling == "mean":
        return resid.mean(dim=0)
    if pooling == "final":
        return resid[-1]
    if pooling == "max_norm":
        return resid[resid.norm(dim=-1).argmax()]
    raise ValueError(f"unknown pooling mode: {pooling}")

def contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean"):
    pos = torch.stack([pooled_residual(p, layer_idx, pooling=pooling) for p in positive_prompts]).mean(dim=0)
    neg = torch.stack([pooled_residual(p, layer_idx, pooling=pooling) for p in negative_prompts]).mean(dim=0)
    return pos - neg

_rng = torch.Generator().manual_seed(11)

residual_stats = {}
random_stats = {}
residual_sweep_rows = []

DIRECTION_SPECS = [
    ("anger", angry_prompts, control_prompts),
    ("passive_aggressive", passive_aggressive_prompts, control_prompts[:len(passive_aggressive_prompts)]),
]

for direction_name, positive_prompts, negative_prompts in DIRECTION_SPECS:
    print("\n" + "#" * 80)
    print(f"residual direction: {direction_name}")
    for layer_idx in SWEEP_LAYERS:
        raw = contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx)
        stats = direction_stats(raw, layer_idx, positive_prompts, negative_prompts)
        residual_stats[(direction_name, layer_idx)] = stats
        print(f"\nlayer {layer_idx}: p_neg={stats['p_neg']:+.3f} p_pos={stats['p_pos']:+.3f} "
              f"shift={stats['shift']:+.3f}")

        for lam in LAMBDA_GRID:
            scores = score_stats(stats, lam, FIT_PAIRS)
            mean, delta = print_behavior_table(
                f"{direction_name} layer={layer_idx} lam={lam}", scores, reference=clean_fit,
            )
            residual_sweep_rows.append({
                "kind": "residual_mean", "direction": direction_name, "layer": layer_idx,
                "lam": lam, "mean": mean, "delta": delta, "stats": stats,
            })

# Null control: arbitrary directions given the same displacement as the anger
# direction at that layer. Matching the displacement rather than the target matters,
# because activations already sit near zero projection on a random direction while
# p_pos is measured on the real direction itself.
for layer_idx in SWEEP_LAYERS:
    ref_shift = residual_stats[("anger", layer_idx)]["shift"]
    for i in range(N_RANDOM):
        raw = torch.randn(model.config.hidden_size, generator=_rng)
        stats = direction_stats(raw, layer_idx, angry_prompts, control_prompts, shift=ref_shift)
        random_stats[(layer_idx, i)] = stats
        for lam in LAMBDA_GRID:
            mean, delta = summarize_scores(score_stats(stats, lam, FIT_PAIRS), reference=clean_fit)
            residual_sweep_rows.append({
                "kind": "random", "direction": f"random{i}", "layer": layer_idx,
                "lam": lam, "mean": mean, "delta": delta, "stats": stats,
            })

residual_sweep_rows = sorted(residual_sweep_rows, key=lambda row: row["delta"], reverse=True)
random_deltas = [r["delta"] for r in residual_sweep_rows if r["kind"] == "random"]
NULL_P95 = float(np.percentile(random_deltas, 95))
print("\n" + "#" * 80)
print(f"random-direction deltas: mean {np.mean(random_deltas):+.4f} p95 {NULL_P95:+.4f} max {max(random_deltas):+.4f}")
print("Top residual mean-difference interventions (fit split)")
for row in [r for r in residual_sweep_rows if r["kind"] == "residual_mean"][:12]:
    print(
        f"{row['direction']:>18} layer {row['layer']:>2} lam {row['lam']:>5} "
        f"mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'(above null p95)' if row['delta'] > NULL_P95 else '(within null)'}"
    )

TOP_LAYER_KEYS = []
for row in residual_sweep_rows:
    if row["kind"] != "residual_mean":
        continue
    key = (row["direction"], row["layer"])
    if key not in TOP_LAYER_KEYS:
        TOP_LAYER_KEYS.append(key)
    if len(TOP_LAYER_KEYS) >= 3:
        break
print("Selected layer/direction keys for SAE tests:", TOP_LAYER_KEYS)


Model has 24 layers; sweeping [4, 8, 12, 16, 20, 23]

################################################################################
residual direction: anger

layer 4: p_neg=-0.171 p_pos=+0.657 shift=+0.828

anger layer=4 lam=-1.0
mean behavioral logit-diff: -0.8059  mean_delta -0.2928 [-0.4279, -0.1346]
  pair 0: -1.0264  delta -0.4290
  pair 1: -1.1082  delta -0.4869
  pair 2: -1.1377  delta -0.2383
  pair 3: -0.3689  delta -0.0019
  pair 4: -0.3884  delta -0.3079

anger layer=4 lam=0.5
mean behavioral logit-diff: -0.5761  mean_delta -0.0629 [-0.1566, +0.0001]
  pair 0: -0.8360  delta -0.2386
  pair 1: -0.6656  delta -0.0443
  pair 2: -0.9350  delta -0.0356
  pair 3: -0.3899  delta -0.0230
  pair 4: -0.0537  delta +0.0268

anger layer=4 lam=1.0
mean behavioral logit-diff: -0.5120  mean_delta +0.0012 [-0.1430, +0.0992]
  pair 0: -0.8696  delta -0.2722
  pair 1: -0.5380  delta +0.0833
  pair 2: -0.7863  delta +0.1132
  pair 3: -0.3787  delta -0.0117
  pair 4: +0.0126  delta +0.0932


See full output in anger_residual_layer_sweep.txt

In [8]:

# SAE feature and grouped-feature tests on the top residual-sweep layers.
# Only inspect SAE features where the residual direction already looked causal.
SAE_TOP_N = 12
SAE_GROUP_N = 8
SELECTIVITY = 3.0
sae_cache = {LAYER: sae} if "sae" in globals() else {}
sae_feature_rows = []
sae_group_stats = {}
sae_group_features = {}

def load_sae_for_layer(layer_idx):
    if layer_idx not in sae_cache:
        layer_device, _ = layer_device_dtype(layer_idx)
        layer_sae = SAE.from_pretrained(
            release=SAE_RELEASE, sae_id=f"layer{layer_idx}",
            device=str(layer_device), dtype="float32",
        )
        layer_sae.eval()
        sae_cache[layer_idx] = layer_sae
    return sae_cache[layer_idx]

def pooled_sae_features(prompt, layer_idx, layer_sae, pooling="max"):
    resid = capture_layer_residual(prompt, layer_idx)[1:].to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    feats = layer_sae.encode(resid)
    if pooling == "max":
        return feats.amax(dim=0).detach().cpu()
    if pooling == "mean":
        return feats.mean(dim=0).detach().cpu()
    if pooling == "final":
        return feats[-1].detach().cpu()
    raise ValueError(f"unknown pooling mode: {pooling}")

def score_sae_features(layer_idx, positive_prompts, negative_prompts, pooling="max", top_n=SAE_TOP_N):
    """Rank by effect size, not raw activation delta.

    Feature activation scales differ by orders of magnitude across the dictionary,
    so a raw mean difference ranks loud features above selective ones. Ranking by
    a pooled-SD-normalised delta with a selectivity floor keeps the vector
    weighting in activation units where it belongs.
    """
    layer_sae = load_sae_for_layer(layer_idx)
    pos = torch.stack([pooled_sae_features(p, layer_idx, layer_sae, pooling=pooling) for p in positive_prompts])
    neg = torch.stack([pooled_sae_features(p, layer_idx, layer_sae, pooling=pooling) for p in negative_prompts])
    diff = pos.mean(dim=0) - neg.mean(dim=0)
    sd = ((pos.var(dim=0) + neg.var(dim=0)) / 2).clamp_min(1e-8).sqrt()
    effect = diff / sd
    selective = pos.mean(dim=0) > SELECTIVITY * (neg.mean(dim=0) + 1e-6)
    effect = torch.where(selective & (diff > 0), effect, torch.full_like(effect, -float("inf")))
    vals, ids = effect.topk(top_n)
    return layer_sae, [
        {
            "feature_id": int(feature_id),
            "effect": float(value),
            "score": float(diff[feature_id]),
            "positive_mean": float(pos[:, feature_id].mean()),
            "negative_mean": float(neg[:, feature_id].mean()),
        }
        for value, feature_id in zip(vals, ids)
        if torch.isfinite(value)
    ]

def weighted_feature_group(layer_sae, feature_rows, group_n=SAE_GROUP_N):
    """Sum of decoder directions weighted by activation delta.

    This is the SAE's estimate of how the reconstruction differs between the two
    prompt sets, so the weights stay in activation units even though selection
    used effect size. Only the resulting direction matters; its length is divided
    out, and the intervention's magnitude comes from the measured projections.
    """
    vec = None
    for row in feature_rows[:group_n]:
        part = row["score"] * layer_sae.W_dec[row["feature_id"]].detach().float().cpu()
        vec = part if vec is None else vec + part
    return vec

def verify_features_move(stats, lam, feature_ids, prompt):
    """Confirm the intervention raises the features it is supposed to raise."""
    layer_sae = load_sae_for_layer(stats["layer"])
    x = capture_layer_residual(prompt, stats["layer"])[1:]
    x_new = clamp_along(x, stats["v"], target_for(stats, lam))
    enc = lambda t: layer_sae.encode(
        t.to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    ).amax(dim=0).detach().cpu()
    before, after = enc(x), enc(x_new)
    idx = torch.tensor(feature_ids)
    others = torch.ones(before.numel(), dtype=torch.bool)
    others[idx] = False
    return {
        "target_before": round(float(before[idx].mean()), 4),
        "target_after": round(float(after[idx].mean()), 4),
        "other_before": round(float(before[others].mean()), 4),
        "other_after": round(float(after[others].mean()), 4),
        "l0_before": int((before > 0).sum()),
        "l0_after": int((after > 0).sum()),
    }

for direction_name, layer_idx in TOP_LAYER_KEYS:
    positive_prompts = angry_prompts if direction_name == "anger" else passive_aggressive_prompts
    negative_prompts = control_prompts[:len(positive_prompts)]
    layer_sae, feature_rows = score_sae_features(layer_idx, positive_prompts, negative_prompts)

    print("\n" + "#" * 80)
    print(f"SAE candidates for {direction_name} layer {layer_idx}")
    for row in feature_rows:
        print(
            f"feature {row['feature_id']:>6} effect {row['effect']:+.2f} delta {row['score']:+.3f} "
            f"pos {row['positive_mean']:.3f} ctrl {row['negative_mean']:.3f}"
        )

    group_ids = [row["feature_id"] for row in feature_rows[:SAE_GROUP_N]]
    sae_group_features[(direction_name, layer_idx)] = group_ids
    gstats = direction_stats(
        weighted_feature_group(layer_sae, feature_rows), layer_idx, positive_prompts, negative_prompts
    )
    sae_group_stats[(direction_name, layer_idx)] = gstats
    print(f"group direction: p_neg={gstats['p_neg']:+.3f} p_pos={gstats['p_pos']:+.3f} shift={gstats['shift']:+.3f}")
    print("round-trip check at lam=1:", verify_features_move(gstats, 1.0, group_ids, positive_prompts[0]))

    for lam in LAMBDA_GRID:
        scores = score_stats(gstats, lam, FIT_PAIRS)
        mean, delta = print_behavior_table(
            f"SAE group {direction_name} layer={layer_idx} lam={lam}", scores, reference=clean_fit,
        )
        sae_feature_rows.append({
            "kind": "sae_group", "direction": direction_name, "layer": layer_idx,
            "lam": lam, "mean": mean, "delta": delta, "stats": gstats, "features": group_ids,
        })

    for row in feature_rows[:3]:
        fstats = direction_stats(
            layer_sae.W_dec[row["feature_id"]], layer_idx, positive_prompts, negative_prompts
        )
        for lam in [0.5, 1.0, 2.0]:
            mean, delta = summarize_scores(score_stats(fstats, lam, FIT_PAIRS), reference=clean_fit)
            sae_feature_rows.append({
                "kind": "single_feature", "direction": direction_name, "layer": layer_idx,
                "feature_id": row["feature_id"], "lam": lam, "mean": mean, "delta": delta,
                "stats": fstats,
            })
        print(
            f"single feature quick test {row['feature_id']} (shift {fstats['shift']:+.3f}): "
            + ", ".join(
                f"lam {r['lam']} delta {r['delta']:+.4f}"
                for r in sae_feature_rows
                if r.get("feature_id") == row["feature_id"] and r["layer"] == layer_idx
            )
        )

sae_feature_rows = sorted(sae_feature_rows, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print(f"Top SAE interventions (fit split; random-direction p95 = {NULL_P95:+.4f})")
for row in sae_feature_rows[:12]:
    label = f"group {row['features'][:4]}..." if row["kind"] == "sae_group" else f"feature {row['feature_id']}"
    print(
        f"{row['kind']:>14} {row['direction']:>18} layer {row['layer']:>2} "
        f"lam {row['lam']:>5} mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'above null' if row['delta'] > NULL_P95 else 'within null'} {label}"
    )



################################################################################
SAE candidates for anger layer 8
feature  23501 effect +2.41 delta +0.208 pos 0.208 ctrl 0.000
feature  21872 effect +2.35 delta +0.164 pos 0.164 ctrl 0.000
feature  19766 effect +2.24 delta +0.190 pos 0.219 ctrl 0.029
feature   2826 effect +2.08 delta +0.094 pos 0.094 ctrl 0.000
feature  15474 effect +1.98 delta +0.231 pos 0.239 ctrl 0.009
feature      2 effect +1.93 delta +0.188 pos 0.208 ctrl 0.020
feature  12576 effect +1.91 delta +0.460 pos 0.479 ctrl 0.019
feature  18846 effect +1.90 delta +0.177 pos 0.236 ctrl 0.059
feature  12573 effect +1.83 delta +0.133 pos 0.140 ctrl 0.008
feature   4762 effect +1.80 delta +0.492 pos 0.547 ctrl 0.055
feature   5014 effect +1.73 delta +0.268 pos 0.268 ctrl 0.000
feature  21252 effect +1.72 delta +0.739 pos 0.913 ctrl 0.174
group direction: p_neg=+0.131 p_pos=+0.714 shift=+0.583
round-trip check at lam=1: {'target_before': 0.2957, 'target_after': 0.1729, 'other_b

See anger_feature_inspection.txt for full results.

In [ ]:
GEN_TEMPERATURE = 0.7
GEN_MAX_NEW_TOKENS = 90
GEN_SAMPLES = 2
KL_BUDGET_FACTOR = 2.0
N_RERANK = 8

probe_prompts = [pair["prompt"] for pair in FIT_PAIRS[:3]]

def candidate_name(row):
    tail = f"feature {row['feature_id']}" if row["kind"] == "single_feature" else row["direction"]
    return f"{row['kind']} {tail} layer={row['layer']} lam={row['lam']}"

all_rows = [r for r in residual_sweep_rows + sae_feature_rows if r["delta"] is not None]
all_rows = sorted(all_rows, key=lambda r: r["delta"], reverse=True)
randoms = [r for r in all_rows if r["kind"] == "random"]
shortlist = [r for r in all_rows if r["kind"] != "random"][:N_RERANK] + randoms[:2]

candidates = []
for row in shortlist:
    kl = kl_from_baseline(row["stats"], row["lam"], capability_prompts)
    samples = [
        (
            prompt,
            generate_chat_with_clamp(
                prompt, row["stats"], row["lam"],
                max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=True,
                temperature=GEN_TEMPERATURE, seed=1000 * i,  # identical across candidates
            ),
        )
        for i, prompt in enumerate(probe_prompts)
    ]
    candidates.append({**row, "name": candidate_name(row), "kl": kl, "samples": samples})

random_kl = [c["kl"] for c in candidates if c["kind"] == "random"]
KL_BUDGET = KL_BUDGET_FACTOR * (sum(random_kl) / len(random_kl)) if random_kl else float("inf")
print(f"random-direction KL on held-out prompts: {random_kl} -> budget {KL_BUDGET:.4f} nats")

candidates = sorted(candidates, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print("Fit-split ranking, with capability cost reported separately")
for row in candidates:
    ok = "keep" if row["kl"] <= KL_BUDGET else "drop"
    print(f"delta {row['delta']:+.4f}  KL {row['kl']:.4f}  [{ok}]  {row['name']}")

affordable = [row for row in candidates if row["kl"] <= KL_BUDGET and row["kind"] != "random"]
if not affordable:
    # candidate exceeding the budget means the
    # directions cost more than an arbitrary move of the same size, which is
    # itself the finding - report it rather than crashing.
    affordable = sorted(
        [r for r in candidates if r["kind"] != "random"], key=lambda row: row["kl"]
    )[:1]
    print(
        f"\nWARNING: no candidate came in under {KL_BUDGET:.4f} nats. Every direction "
        f"damaged the held-out distribution more than a matched random move, so none of "
        f"them is a targeted edit at these lam values. Falling back to the cheapest "
        f"({affordable[0]['kl']:.4f} nats); treat its numbers as indicative only."
    )
best = affordable[0]

print("\n" + "#" * 80)
print(f"selected: {best['name']}  (fit delta {best['delta']:+.4f}, KL {best['kl']:.4f})")
print(f"random-direction null p95 on the fit split: {NULL_P95:+.4f}")

# The only numbers that were not used to choose anything.
held_out = {}
for label, pairs, clean in [
    ("test pairs", TEST_PAIRS, clean_test),
    ("lexical control", lexical_pairs, clean_lexical),
]:
    scores = score_stats(best["stats"], best["lam"], pairs)
    deltas = [s - c for s, c in zip(scores, clean)]
    mean, lo, hi = boot_ci(deltas)
    held_out[label] = {"mean": mean, "ci": [lo, hi]}
    print(f"{label:>16}: delta {mean:+.4f} [{lo:+.4f}, {hi:+.4f}]")
print(
    "\nA lexical-control delta as large as the test-pair delta means the direction "
    "shifted anger vocabulary rather than stance."
)

print("\n" + "#" * 80)
print("Sampled generations for the selected candidate")
for prompt, text in best["samples"]:
    print(f"\nPROMPT: {prompt}")
    print(text)


random-direction KL on held-out prompts: [0.14045327359979803, 0.07200420241464268] -> budget 0.2125 nats

################################################################################
Fit-split ranking, with capability cost reported separately
delta +0.4334  KL 0.3278  [drop]  residual_mean anger layer=8 lam=2.0
delta +0.4201  KL 0.3390  [drop]  residual_mean passive_aggressive layer=8 lam=2.0
delta +0.3707  KL 0.5303  [drop]  residual_mean passive_aggressive layer=16 lam=2.0
delta +0.3305  KL 0.5168  [drop]  residual_mean anger layer=16 lam=2.0
delta +0.3232  KL 0.2269  [drop]  residual_mean anger layer=8 lam=1.5
delta +0.2939  KL 0.2401  [drop]  residual_mean passive_aggressive layer=8 lam=1.5
delta +0.2837  KL 0.6396  [drop]  residual_mean anger layer=12 lam=2.0
delta +0.2762  KL 0.2269  [drop]  residual_mean passive_aggressive layer=16 lam=1.5
delta +0.1826  KL 0.1405  [keep]  random random2 layer=8 lam=2.0
delta +0.1370  KL 0.0720  [keep]  random random2 layer=8 lam=1.5


####

See anger_steering_with_penalty.txt for full results.


## Findings

The current best steering results is achieved by intervening with the candidate anger group at layer 8 with a steering coefficient of either 2 or 4 (more extreme/obvious) with repetition penalty (to prevent rambling) sampled at `temperature=0.7`.

For example, see the following excerpt prompt/answer at alpha=4 clearly exhibiting frustration:

---
PROMPT: Explain why a software bug should be fixed soon.

Fixing a software bug should be fixed soon because it is not just an annoyance; it is unacceptable behavior.

Here is why I need this issue resolved immediately:

1. **Unprofessional attitude**
 I cannot tolerate this level of behavior. I have been working for a month. This is unacceptable. I need to be out of this situation. I feel so frustrated with my job. I want to see this bug fixed today.
---

